## L1 and L2 Regularization

Regularization methods are those which reduce generalization error. We do this by calculating a penalty added to the loss value to penalize model for large weights and biases.

Large weights might indicate that a neuron is attempting to memorize a data element.

### L1 Regularization

L1 regularization's penalty is the sum of all the absolute values for the weights and biases. This is a linear panelty as regularization loss returned by this function is directly proportional to parameter values.

$L_{1w} = \lambda \sum_m |w_m|$

### L2 Regularization

L2's penalty is the sum of the squared weights and biases. This non-linear approach penalizes larger weights and biases more than the smaller ones. 

$L_{2w} = \lambda \sum_m w^2_m$

L2 is more commanly used as it does not effect small paramater values substantially and does not allow the model to grow weights too large by heavily penalizing relatively big values.
Regularization also help with the problems of exploding gradients.

Overall Loss:- $Loss\ = \ DataLoss + L_{1w} + L_{1b} + L_{2w} + L_{2b}$

In [1]:
class Dense_layer:

    def __init__(self, n_inputs, n_neurons, weight_regularizer_l1 = 0, weight_regularizer_l2 = 0, bias_regularizer_l1 = 0, bias_regularizer_l2 = 0):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))

        # set regularization penalty
        self.weight_regularizer_l1 = weight_regularizer_l1
        self.weight_regularizer_l2 = weight_regularizer_l2
        self.bias_regularizer_l1 = bias_regularizer_l1
        self.bias_regularizer_l2 = bias_regularizer_l2
    
    # we want to remember our input to calculate gradient of weights
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases

    
    def backward(self, dvalues):
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis = 0, keepdims = True)
        self.dinputs = np.dot(dvalues, self.weights.T)

In [2]:
class Loss:
    def regularization_loss(self, layer):

        # 0 by default
        regularization_loss = 0

        # L1 regularization - weights
        if layer.weight_regularizer_l1 > 0:
            regularization_loss += layer.weight_regularizer_l1 * np.sum(np.abs(layer.weights))

        # L2 regularization - weights
        if layer.weight_regularizer_l2 > 0:
            regularization_loss += layer.weight_regularizer_l2 * np.sum(layer.weights * layer.weights)

        # L1 regularization - biases
        if layer.bias_regularizer_l1 > 0:
            regularization_loss += layer.bias_regularizer_l1 * np.sum(np.abs(layer.biases))

        # L2 regularization - biases
        if layer.bias_regularizer_l2 > 0:
            regularization_loss += layer.bias_regularizer_l2 * np.sum(layer.biases * layer.biases)

        return regularization_loss

    def calculate(self, output, y):
        #calculate sample losses 
        sample_losses = self.forward(output, y)

        data_loss = np.mean(sample_losses)

        return data_loss

We will now have to add the backpropogation part as well. For that we need partial derivative of L1 and L2 regularization.

Derivative of L2 is given as $2 \lambda w_m$. 

Derivative of L1 is given as:-

$\lambda\ \ if\ \ w_m > 0\ \ or$

$-\lambda\ \ if\ \ w_m < 0\ \ $

In [3]:
weights = [0.2, 0.8, -0.5]

dL1 = []
for weight in weights:
    if weight >= 0:
        dL1.append(1)
    else:
        dL1.append(-1)

print(dL1)

[1, 1, -1]


In [4]:
class Dense_layer:

    def __init__(self, n_inputs, n_neurons, weight_regularizer_l1 = 0, weight_regularizer_l2 = 0, bias_regularizer_l1 = 0, bias_regularizer_l2 = 0):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))

        # set regularization penalty
        self.weight_regularizer_l1 = weight_regularizer_l1
        self.weight_regularizer_l2 = weight_regularizer_l2
        self.bias_regularizer_l1 = bias_regularizer_l1
        self.bias_regularizer_l2 = bias_regularizer_l2
    
    # we want to remember our input to calculate gradient of weights
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases

    
    def backward(self, dvalues):
        #Gradients on parameter
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis = 0, keepdims = True)

        #Gradients on regularization
        # L1 on weights
        if self.weight_regularizer_l1 > 0:
            dL1 = np.ones_like(self.weights)
            dL1[self.weights < 0] = -1
            self.dweights += self.weight_regularizer_l1 * dL1

        # L2 on weights
        if self.weight_regularizer_l2 > 0:
            self.dweights += 2 * self.weight_regularizer_l2 * self.weights

        # L1 on biases
        if self.bias_regularizer_l1 > 0:
            dL1 = np.ones_like(self.biases)
            dL1[self.biases < 0] = -1
            self.dbiases += self.bias_regularizer_l1 * dL1  

        # L2 on biases
        if self.bias_regularizer_l2 > 0:
            self.dbiases += 2 * self.bias_regularizer_l2 * self.biases       
        
        self.dinputs = np.dot(dvalues, self.weights.T)

class Activation_Relu:

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0


class Activation_softmax:
    
    def forward(self, inputs):

        exp_values = np.exp(inputs - np.max(inputs, axis = 1, keepdims = True))
        probabilities = exp_values/np.sum(exp_values, axis = 1, keepdims = True)
        self.output = probabilities

    def backward(self, dvalues):
        self.dinputs = np.empty_like(dvalues)

        for index, (single_output, single_dvalues) in enumerate(zip(self.output, dvalues)):
            single_output = single_output.reshape(-1,1)

            jacobian_matrix = np.diagflat(single_output) - np.dot(single_output, single_output.T)

            #calculate sample-wise gradient
            #and add it to the array of sample gradients
            self.dinputs[index] = np.dot(jacobian_matrix, single_dvalues)

class Loss:
    def regularization_loss(self, layer):

        # 0 by default
        regularization_loss = 0

        # L1 regularization - weights
        if layer.weight_regularizer_l1 > 0:
            regularization_loss += layer.weight_regularizer_l1 * np.sum(np.abs(layer.weights))

        # L2 regularization - weights
        if layer.weight_regularizer_l2 > 0:
            regularization_loss += layer.weight_regularizer_l2 * np.sum(layer.weights * layer.weights)

        # L1 regularization - biases
        if layer.bias_regularizer_l1 > 0:
            regularization_loss += layer.bias_regularizer_l1 * np.sum(np.abs(layer.biases))

        # L2 regularization - biases
        if layer.bias_regularizer_l2 > 0:
            regularization_loss += layer.bias_regularizer_l2 * np.sum(layer.biases * layer.biases)

        return regularization_loss

    def calculate(self, output, y):
        #calculate sample losses 
        sample_losses = self.forward(output, y)

        data_loss = np.mean(sample_losses)

        return data_loss
    
class Loss_CategoricalCrossentropy(Loss):

    def forward(self, y_pred, y_true):

        samples = len(y_pred)

        #We are capping the y_pred for preventing log(0) or log(high value) both are not defined
        y_pred_clipped = np.clip(y_pred, 1e-7, 1- 1e-7)

        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[range(samples), y_true]

        if len(y_true.shape) == 2:
            correct_confidences = np.sum(y_pred_clipped * y_true, axis = 1)

        negative_log_likelihood = -np.log(correct_confidences)
        return negative_log_likelihood
    
    def backward(self, dvalues, y_true):
        samples = len(dvalues) # Number of samples

        #Number of labels in every sample
        #We'll use the first sample to count them
        labels = len(dvalues[0])

        #If labels are sparse then, turn then into one-hot vector
        if len(y_true.shape) == 1:
            y_true =   np.eye(labels)[y_true]

        #calculate gradient
        self.dinputs = -y_true / dvalues

        #Normalize gradient
        self.dinputs = self.dinputs / samples

class Activation_Softmax_Loss_CategoricalCrossentropy():

    def __init__(self):
        self.activation = Activation_softmax()
        self.loss = Loss_CategoricalCrossentropy()

    # Forward pass
    def forward(self, inputs, y_true):
        #output layer activation function
        self.activation.forward(inputs)

        #set the output
        self.output = self.activation.output

        #calculate and return loss value
        return self.loss.calculate(self.output, y_true)
    
    def backward(self, dvalues, y_true):

        samples = len(dvalues)

        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true, axis = 1)

        self.dinputs = dvalues.copy()
        self.dinputs[range(samples), y_true] -= 1
        self.dinputs = self.dinputs/samples

In [5]:
class Optimizer_Adagrad:

    def __init__(self, learning_rate = 1.0, decay = 0, epsilon = 1e-7):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * (1/(1+ self.decay * self.iterations))


    #update parameters
    def update_params(self, layer):

        if not hasattr(layer, 'weight_cache'):
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_cache = np.zeros_like(layer.biases)

        # Update cache with squared current gradients
        layer.weight_cache += layer.dweights ** 2
        layer.bias_cache += layer.dbiases ** 2

        layer.weights += -self.current_learning_rate * layer.dweights / (np.sqrt(layer.weight_cache) + self.epsilon)
        layer.biases += -self.current_learning_rate * layer.dbiases / (np.sqrt(layer.bias_cache) + self.epsilon)

    def post_update_params(self):
        self.iterations += 1

In [6]:
class Optimizer_RMSProp:

    def __init__(self, learning_rate = 1.0, decay = 0, epsilon = 1e-7, rho = 0.9):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon
        self.rho = rho

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * (1/(1+ self.decay * self.iterations))


    #update parameters
    def update_params(self, layer):

        if not hasattr(layer, 'weight_cache'):
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_cache = np.zeros_like(layer.biases)

        # Update cache with squared current gradients
        layer.weight_cache += self.rho * layer.weight_cache + (1 - self.rho) * layer.dweights ** 2
        layer.bias_cache += self.rho * layer.bias_cache + (1 - self.rho) * layer.dbiases ** 2

        layer.weights += -self.current_learning_rate * layer.dweights / (np.sqrt(layer.weight_cache) + self.epsilon)
        layer.biases += -self.current_learning_rate * layer.dbiases / (np.sqrt(layer.bias_cache) + self.epsilon)

    def post_update_params(self):
        self.iterations += 1

In [7]:
class Optimizer_Adam:

    def __init__(self, learning_rate = 0.001, decay = 0, epsilon = 1e-7, beta_1 = 0.9, beta_2 = 0.999):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon
        self.beta_1 = beta_1
        self.beta_2 = beta_2

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * (1/(1+ self.decay * self.iterations))


    #update parameters
    def update_params(self, layer):

        if not hasattr(layer, 'weight_cache'):
            layer.weight_momentums = np.zeros_like(layer.weights)
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_momentums = np.zeros_like(layer.biases)
            layer.bias_cache = np.zeros_like(layer.biases)

        #Update moementum with current gradients
        layer.weight_momentums = self.beta_1 * layer.weight_momentums + (1 - self.beta_1) * layer.dweights
        layer.bias_momentums = self.beta_1 * layer.bias_momentums + (1 - self.beta_1) * layer.dbiases

        weight_momentums_corrected = layer.weight_momentums / (1 - self.beta_1 ** (self.iterations + 1))
        bias_momentums_corrected = layer.bias_momentums / (1 - self.beta_1 ** (self.iterations + 1))

        # Update cache with squared current gradients
        layer.weight_cache = self.beta_2 * layer.weight_cache + (1 - self.beta_2) * layer.dweights ** 2
        layer.bias_cache += self.beta_2 * layer.bias_cache + (1 - self.beta_2) * layer.dbiases ** 2

        #Get corrected cache
        weight_cache_corrected = layer.weight_cache / (1 - self.beta_2 ** (self.iterations + 1))
        bias_cache_corrected = layer.bias_cache / (1 - self.beta_2 ** (self.iterations + 1))

        layer.weights += -self.current_learning_rate * weight_momentums_corrected / (np.sqrt(weight_cache_corrected) + self.epsilon)
        layer.biases += -self.current_learning_rate * bias_momentums_corrected / (np.sqrt(bias_cache_corrected) + self.epsilon)

    def post_update_params(self):
        self.iterations += 1

In [8]:
import numpy as np
from nnfs.datasets import vertical_data, spiral_data
import nnfs

In [9]:
X, y = spiral_data(samples = 1000, classes = 3)

# Dense layer with 2 input and 64 output values (64 neurons).
dense1 = Dense_layer(2, 512, weight_regularizer_l2=5e-4, bias_regularizer_l2=5e-4)

activation1 = Activation_Relu()

#Create second dense layer that take output from previous layer hence 3 inputs and 3 output values
dense2 = Dense_layer(512, 3)

#We are now using new class to calculate final softmax activation and loss calculation.
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

In [10]:
optimizer = Optimizer_Adam(decay = 5e-7, learning_rate=0.02)

for epoch in range (10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)
    data_loss = loss_activation.forward(dense2.output, y)

    regularization_loss = loss_activation.loss.regularization_loss(dense1) + loss_activation.loss.regularization_loss(dense2)

    loss = data_loss + regularization_loss

    predictions = np.argmax(loss_activation.output, axis = 1)
    if len(y.shape) == 2:
        y = np.argmax(y, axis = 1)
    accuracy = np.mean(predictions == y)

    if not epoch % 100:
        print(f'epoch: {epoch}, ' +
              f'acc: {accuracy:.3f}, ' +
              f'loss: {loss:.3f}, ' +
              f'data_loss: {data_loss:.3f}, ' +
              f'reg_loss: {regularization_loss:.3f}, ' +
              f'lr: {optimizer.current_learning_rate}')

    loss_activation.backward(loss_activation.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()

epoch: 0, acc: 0.338, loss: 1.099, data_loss: 1.099, reg_loss: 0.000, lr: 0.02
epoch: 100, acc: 0.532, loss: 0.982, data_loss: 0.979, reg_loss: 0.004, lr: 0.019999010049002574
epoch: 200, acc: 0.613, loss: 0.918, data_loss: 0.914, reg_loss: 0.005, lr: 0.019998010197985302
epoch: 300, acc: 0.670, loss: 0.861, data_loss: 0.856, reg_loss: 0.005, lr: 0.019997010446938183
epoch: 400, acc: 0.676, loss: 0.832, data_loss: 0.827, reg_loss: 0.005, lr: 0.01999601079584623
epoch: 500, acc: 0.718, loss: 0.797, data_loss: 0.792, reg_loss: 0.005, lr: 0.01999501124469445
epoch: 600, acc: 0.721, loss: 0.777, data_loss: 0.771, reg_loss: 0.006, lr: 0.01999401179346786
epoch: 700, acc: 0.723, loss: 0.753, data_loss: 0.748, reg_loss: 0.006, lr: 0.01999301244215147
epoch: 800, acc: 0.769, loss: 0.733, data_loss: 0.727, reg_loss: 0.006, lr: 0.0199920131907303
epoch: 900, acc: 0.770, loss: 0.715, data_loss: 0.709, reg_loss: 0.006, lr: 0.019991014039189386
epoch: 1000, acc: 0.796, loss: 0.696, data_loss: 0.690

<ipython-input-7-6000be9e7400>:39: RuntimeWarning: overflow encountered in true_divide
  bias_cache_corrected = layer.bias_cache / (1 - self.beta_2 ** (self.iterations + 1))
<ipython-input-7-6000be9e7400>:35: RuntimeWarning: overflow encountered in add
  layer.bias_cache += self.beta_2 * layer.bias_cache + (1 - self.beta_2) * layer.dbiases ** 2


epoch: 1100, acc: 0.790, loss: 0.690, data_loss: 0.683, reg_loss: 0.006, lr: 0.01998901603568839
epoch: 1200, acc: 0.766, loss: 0.693, data_loss: 0.687, reg_loss: 0.006, lr: 0.019988017183698373
epoch: 1300, acc: 0.806, loss: 0.656, data_loss: 0.649, reg_loss: 0.007, lr: 0.01998701843152872
epoch: 1400, acc: 0.811, loss: 0.647, data_loss: 0.641, reg_loss: 0.007, lr: 0.019986019779164473
epoch: 1500, acc: 0.807, loss: 0.633, data_loss: 0.626, reg_loss: 0.007, lr: 0.019985021226590672
epoch: 1600, acc: 0.804, loss: 0.623, data_loss: 0.616, reg_loss: 0.007, lr: 0.01998402277379235
epoch: 1700, acc: 0.826, loss: 0.609, data_loss: 0.602, reg_loss: 0.007, lr: 0.01998302442075457
epoch: 1800, acc: 0.811, loss: 0.605, data_loss: 0.598, reg_loss: 0.007, lr: 0.019982026167462367
epoch: 1900, acc: 0.795, loss: 0.608, data_loss: 0.600, reg_loss: 0.007, lr: 0.019981028013900805
epoch: 2000, acc: 0.793, loss: 0.643, data_loss: 0.636, reg_loss: 0.007, lr: 0.019980029960054924
epoch: 2100, acc: 0.834,

epoch: 9500, acc: 0.881, loss: 0.386, data_loss: 0.376, reg_loss: 0.010, lr: 0.01990545902237324
epoch: 9600, acc: 0.883, loss: 0.384, data_loss: 0.374, reg_loss: 0.010, lr: 0.019904468503417844
epoch: 9700, acc: 0.892, loss: 0.387, data_loss: 0.377, reg_loss: 0.010, lr: 0.019903478083036316
epoch: 9800, acc: 0.798, loss: 0.580, data_loss: 0.570, reg_loss: 0.010, lr: 0.019902487761213932
epoch: 9900, acc: 0.883, loss: 0.387, data_loss: 0.378, reg_loss: 0.010, lr: 0.019901497537935988
epoch: 10000, acc: 0.881, loss: 0.385, data_loss: 0.375, reg_loss: 0.010, lr: 0.019900507413187767


In [11]:
X_test, y_test = spiral_data(samples = 100, classes = 3)
dense1.forward(X_test)
activation1.forward(dense1.output)
dense2.forward(activation1.output)
loss = loss_activation.forward(dense2.output, y_test)

predictions = np.argmax(loss_activation.output, axis = 1)
if len(y_test.shape) == 2:
    y_test = np.argmax(y_test, axis = 1)
accuracy = np.mean(predictions == y_test)

print(f'validation, acc: {accuracy:.3f}, loss: {loss:.3f}')

validation, acc: 0.883, loss: 0.416
